# 02 - Data Cleaning (Stage 3)

Turn the raw, all-text CSV into a clean, typed dataset. The logic lives in `src/amazon_project/cleaning.py`; this notebook calls it step by step and checks each result. Raw data is never modified: output goes to `data/processed/amazon_clean.csv`.

In [ ]:
from amazon_project.cleaning import (
    add_review_count,
    clean_data,
    convert_numeric_columns,
    drop_duplicate_products,
    split_category,
    strip_whitespace,
)
from amazon_project.loading import load_processed, load_raw, save_processed

raw = load_raw()
raw.shape

## 1. Confirm the known problems
Before fixing anything, check the quirks documented in CLAUDE.md are really there.

In [ ]:
print("Empty cells:", (raw == "").sum()[lambda s: s > 0].to_dict())
print(
    "Invalid ratings:",
    raw.loc[~raw["rating"].str.fullmatch(r"\d(\.\d)?"), "rating"].tolist(),
)
print("Extra rows with a repeated product_id:", raw["product_id"].duplicated().sum())

## 2-3. Whitespace, missing values and numeric conversion
`₹1,099` -> `1099.0`, `64%` -> `64`, `24,269` -> `24269`. Anything that cannot become a number (the empty `rating_count` cells and the `|` rating) becomes a real missing value (NaN). We keep them missing rather than inventing numbers.

In [ ]:
df = convert_numeric_columns(strip_whitespace(raw))
df[
    [
        "discounted_price",
        "actual_price",
        "discount_percentage",
        "rating",
        "rating_count",
    ]
].dtypes

In [ ]:
df.isna().sum()[lambda s: s > 0]

## 4. Validate the numbers
Do the values make sense? Discounted price should never exceed the list price, ratings must be 0-5, and the stated discount should match what the two prices imply.

In [ ]:
print("Discounted > actual:", (df["discounted_price"] > df["actual_price"]).sum())
print("Prices <= 0:", ((df["discounted_price"] <= 0) | (df["actual_price"] <= 0)).sum())
print("Rating range:", df["rating"].min(), "to", df["rating"].max())
implied = (1 - df["discounted_price"] / df["actual_price"]) * 100
print(
    "Largest gap between stated and implied discount (points):",
    (implied - df["discount_percentage"]).abs().max(),
)

All checks pass; the largest discount gap (0.5 points) is just rounding. Outliers are noted but **not** removed - that is an analysis decision for Stage 4.

## 5. Duplicate product IDs
The 114 repeated IDs are the same product scraped from different search-result pages (links differ only in `ref=sr_1_N`). Category and rating match within each group; `rating_count` differs by at most 13. **Rule:** keep one row per `product_id`, the one with the highest `rating_count`.

In [ ]:
deduped = drop_duplicate_products(df)
print(
    len(df), "->", len(deduped), "rows;", "unique ids:", deduped["product_id"].is_unique
)

## 6. Split the category hierarchy
`main_category`, `sub_category`, `leaf_category`, plus `is_small_category` marking main categories with fewer than 5 products.

In [ ]:
cats = split_category(deduped)
cats["main_category"].value_counts()

## 7. Review columns
IDs split safely on commas (user_id and review_id counts match in every row), but titles and review text contain commas, so they cannot be split reliably. **Decision:** keep review text as raw text and add only `n_reviews`, counted from `review_id`.

In [ ]:
final = add_review_count(cats).reset_index(drop=True)
final["n_reviews"].describe()

## 9. Save and reload
Saved as CSV (git-ignored). CSV forgets column types, so `load_processed()` restores them.

In [ ]:
# The one-call pipeline must give the same result as the steps above.
assert clean_data(raw).equals(final)
path = save_processed(final)
check = load_processed(path)
print(path.name, check.shape)
check.dtypes